***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [7. 观测系统](#)
    * 下一节： [7.1 琼斯符号](7_1_jones_notation.ipynb)

***

导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path('7_Observing_Systems') if Path('7_Observing_Systems').exists() else Path('.')
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'course.css'
TOGGLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'code_toggle.html'

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

本章不再把望远镜视为一个“黑盒子”。从这里开始，我们要把信号从天空传播到相关器输出的整条链路拆开来看：电磁波如何用 Jones 向量与 Jones 矩阵描述，传播介质与仪器部件怎样改变它，模拟链路和数字相关器怎样把连续波前变成可存储、可校准的复可见度，以及这些系统效应为什么最终会出现在图像和源表里。

和前面几章相比，这一章的目标更偏“观测系统建模”。它不要求你立刻记住所有硬件细节，但要求你建立一个非常稳固的判断：任何测量都会选择性保留信息、丢失信息并引入噪声；而校准与成像的工作，本质上就是尽可能把这些系统效应从科学信号里分离出来。

In [ ]:
if TOGGLE_PATH.exists():
    try:
        HTML(TOGGLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

***

# 第七章：观测系统 <a id='instrum:sec:intro'></a>

如果把综合孔径阵列想成一个端到端的信号处理系统，那么它至少包含四层互相耦合的结构：

1. 天空与传播介质决定了“输入信号长什么样”；
2. 天线、馈源和模拟链路决定了“输入怎样被接收和放大”；
3. ADC、滤波器组与相关器决定了“连续信号怎样变成离散相关量”；
4. 校准与成像决定了“这些相关量最后怎样回到科学上可解释的天空模型”。

因此，本章真正关心的不是孤立的硬件名词，而是“哪一类系统效应在什么数学对象里出现”。比如：方向无关增益会进入 Jones 链中的增益项；主波束与传播介质会引入方向相关效应；相关器结构则决定了时间、频率与数据率之间的工程折中。只有把这些位置关系理清楚，后面第 8 章的校准才不会显得像一串经验参数。

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.2))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')

blocks = [
    (0.5, 1.4, 2.0, 1.2, '#d9f0ff', 'Sky Signal', '7.1-7.2'),
    (3.0, 1.4, 2.2, 1.2, '#e8f7d4', 'Propagation', '7.7'),
    (5.8, 1.4, 2.0, 1.2, '#fff0c9', 'Antenna / Feed', '7.5-7.6'),
    (8.3, 1.4, 2.0, 1.2, '#fde0c5', 'Analog Chain', '7.3'),
    (10.8, 1.4, 2.0, 1.2, '#f4d9ff', 'Digital Correlator', '7.4'),
]

for x, y, w, h, color, title, subtitle in blocks:
    patch = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08,rounding_size=0.09',
                           linewidth=1.2, edgecolor='0.35', facecolor=color)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + 0.73, title, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(x + w / 2, y + 0.33, subtitle, ha='center', va='center', fontsize=10, color='0.35')

for left, right in [(2.5, 3.0), (5.2, 5.8), (7.8, 8.3), (10.3, 10.8)]:
    arrow = FancyArrowPatch((left, 2.0), (right, 2.0), arrowstyle='-|>', mutation_scale=16,
                            linewidth=1.5, color='0.35')
    ax.add_patch(arrow)

ax.text(6.9, 3.3, 'Visibility-domain system model', ha='center', fontsize=13, fontweight='bold')
ax.text(12.8, 0.55, 'Calibration & imaging in Chapter 8', ha='center', fontsize=11, color='0.35')
arrow = FancyArrowPatch((12.7, 1.3), (12.7, 0.75), arrowstyle='-|>', mutation_scale=15,
                        linewidth=1.3, color='0.35')
ax.add_patch(arrow)

plt.tight_layout()

*图：第 7 章的主线不是逐个介绍零散器件，而是把“天空信号如何经过传播、接收、放大、数字化与相关，最终变成可校准的可见度”这条链路串起来。这里的章节编号对应本章后续各节。*

#### 本章目录

1. [琼斯符号](7_1_jones_notation.ipynb)：建立信号、极化和线性系统效应的统一数学语言。
2. [射电干涉测量方程（RIME）](7_2_rime.ipynb)：把 Jones 记号正式接到相关矩阵和可见度方程。
3. [模拟电子学](7_3_analogue.ipynb)：介绍放大、噪声、带宽和模拟链路中的主要系统限制。
4. [数字相关器](7_4_digital.ipynb)：讨论采样、FFT/PFB、FX/XF 结构以及数据率折中。
5. [主波束](7_5_primary_beam.ipynb)：说明方向相关增益为什么会直接影响成像与校准。
6. [极化与天线馈源](7_6_polarization.ipynb)：进一步讨论馈源、极化泄漏与 Mueller/Jones 表达。
7. [传播效应](7_7_propagation_effects.ipynb)：把电离层、对流层和星际介质效应放回观测链路。
8. [射频干扰（RFI）](7_8_rfi.ipynb)：介绍非天体信号怎样进入数据，以及为什么它是现代观测系统的核心挑战之一。

建议的学习顺序是：先把 7.1 和 7.2 中的数学对象彻底弄清，再去读 7.3 到 7.8 中具体的系统效应。否则，很容易把后面的每一种仪器项都学成彼此孤立的经验名词。

***

* 下一节： [7.1 琼斯符号](7_1_jones_notation.ipynb)